## Compare Our Results with Paparella

Note, that we have removed empty (padding only) sequences from test, train and val  
datasets and therefore our NDCG and HR values are better. Moreover, we have modified  
repetitivness metric. Therefore, repetitivnes is not directly comparable with  
Paparellas results.

In [6]:
import os

models = ['sasrec', 'caser', 'gru', 'nextitnet']
combs = [[1,1,1],[1,1,0],[1,0,1],[0,1,1],[0,1,0],[0,0,1]]
model_stop = ["main", "target"]
datasets = ["rc15_results"]
results_dir = "/home/marek/Kinit/my_smorl/Plots/"
os.makedirs(results_dir, exist_ok=True)

In [7]:
import pandas as pd
from matplotlib import pyplot as plt

In [14]:
def plot_helper(ax, df0, df1, df2, df3, PLOT, window=1):
    ax.set_ylabel(PLOT, color='black')
    ax.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='black', linewidth=1, label='Paparella base')
    ax.plot(df1['steps'], df1[PLOT].rolling(window).mean(), color='green', linewidth=1, label='Paparella RL')
    ax.plot(df2['steps'], df2[PLOT].rolling(window).mean(), color='black', linestyle=":", linewidth=2, label='Havrila base')
    ax.plot(df3['steps'], df3[PLOT].rolling(window).mean(), color='green', linestyle=":", linewidth=2, label='Havrila RL')
    
def plot_metrics(basepath, dataset, model, replica, testorval, window=1):  
    dataline = "metrics"   
    df0 = pd.read_pickle(f"{basepath}/Paparella_both/{dataset}/{model}/base_{dataline}")
    df1 = pd.read_pickle(f"{basepath}/Paparella_both/{dataset}/{model}/rl_111_{replica}_{dataline}")
    df2 = pd.read_pickle(f"{basepath}/Havrila_both/{dataset}/{model}/base_{dataline}")
    df3 = pd.read_pickle(f"{basepath}/Havrila_both/{dataset}/{model}/rl_111_{replica}_{dataline}")
    
    #~~~~~~ Plot cov + nov values ~~~~~~
    fig, axs = plt.subplots(3, 2, figsize=(14, 14))
    fig.suptitle(f'Paparella vs Havrila : {model}-{dataset}-{replica}-{testorval}', fontsize=12, y=0.94)
     
    PLOT = f"cov_{testorval}_10"
    ax1 = axs[0, 0]
    ax1.set_title("COV 10", fontsize=10, pad=10)
    ax1.set_ylim(0, 0.8)
    ax1.set_ylabel(PLOT, color='black')
    plot_helper(ax1, df0, df1, df2, df3, PLOT)
    
    PLOT = f"nov_{testorval}_10"
    ax2 = axs[0, 1]
    ax2.set_ylim(0, 0.8)
    ax2.set_title("NOV 10", fontsize=10, pad=10)
    plot_helper(ax2, df0, df1, df2, df3, PLOT)
   
    PLOT = f"hr_{testorval}_10"
    ax3 = axs[1, 0]
    ax3.set_ylim(0.35, 0.6)
    ax3.set_title("HR 10", fontsize=10, pad=10)
    plot_helper(ax3, df0, df1, df2, df3, PLOT)
    
    PLOT = f"ndcg_{testorval}_10"
    ax4 = axs[1, 1]
    ax4.set_title("NDCG 10", fontsize=10, pad=10)
    ax4.set_ylim(0.2, 0.35)
    plot_helper(ax4, df0, df1, df2, df3, PLOT)
    
    PLOT = f"rep_{testorval}_5"
    ax5 = axs[2, 0]
    ax5.set_ylim(5, 20)
    ax5.set_title("REP 5", fontsize=10, pad=10)
    plot_helper(ax5, df0, df1, df2, df3, PLOT)
    
    # plot Loss
    dataline = "loss"
    dfl0 = pd.read_pickle(f"{basepath}/Paparella_both/{dataset}/{model}/base_{dataline}")
    dfl1 = pd.read_pickle(f"{basepath}/Paparella_both/{dataset}/{model}/rl_111_{replica}_{dataline}")
    dfl2 = pd.read_pickle(f"{basepath}/Havrila_both/{dataset}/{model}/base_{dataline}")
    dfl3 = pd.read_pickle(f"{basepath}/Havrila_both/{dataset}/{model}/rl_111_{replica}_{dataline}")
    
    PLOT = f"loss"
    ax6 = axs[2, 1]
    ax6.set_ylim(3, 10)
    ax6.set_title("Loss", fontsize=10, pad=10)
    plot_helper(ax6, dfl0, dfl1, dfl2, dfl3, PLOT, window=10) 
    
    for i, ax in enumerate(axs.flat):
        ax.yaxis.grid(True, color='lightgray', linewidth=0.5)
        ax.tick_params(axis='y', labelsize=8)
    
    handles, labels = ax1.get_legend_handles_labels()
    seen = set()
    unique = [(h, l) for h, l in zip(handles, labels) if not (l in seen or seen.add(l))]
    fig.legend(*zip(*unique), loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.02), fontsize='medium')
    fig.savefig(f"{basepath}/Pap_vs_Hav-{model}-{dataset}-{replica}-{testorval}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)

basepath = "/home/marek/Kinit/my_smorl/Plots"
for dataset in datasets:
    for replica in ['main', 'target']:
        for variant in ['test', 'val']:
            for model in models:
                plot_metrics(basepath, dataset, model, replica, variant)